# Loan Financial Analysis - ETL & Snowflake Schema.

#  1

In [ ]:
import pandas as pd

df = pd.read_csv("Loan.csv")
print(df.shape)
df.head()

## 2

In [ ]:
print(df.isnull().sum())
print()
print(df.dtypes)

## 3

In [ ]:
df['ApplicationDate'] = pd.to_datetime(df['ApplicationDate'])

df['Year']    = df['ApplicationDate'].dt.year
df['Month']   = df['ApplicationDate'].dt.month
df['Quarter'] = df['ApplicationDate'].dt.quarter

df[['ApplicationDate','Year','Month','Quarter']].head()

## 4. 

In [ ]:
def credit_category(score):
    if score >= 750: return 'Excellent'
    elif score >= 700: return 'Good'
    elif score >= 650: return 'Fair'
    else: return 'Poor'

df['CreditCategory'] = df['CreditScore'].apply(credit_category)

def loan_size(amount):
    if amount < 10000: return 'Small'
    elif amount < 30000: return 'Medium'
    else: return 'Large'

df['LoanSizeCategory'] = df['LoanAmount'].apply(loan_size)

def age_group(age):
    if age < 30: return 'Young'
    elif age < 45: return 'Middle'
    elif age < 60: return 'Senior'
    else: return 'Elder'

df['AgeGroup'] = df['Age'].apply(age_group)

df[['CreditScore','CreditCategory','LoanAmount','LoanSizeCategory','Age','AgeGroup']].head()

## 5.

In [ ]:
def risk_level(row):
    score = 0
    if row['CreditScore'] < 600: score += 3
    elif row['CreditScore'] < 700: score += 1
    if row['RiskScore'] > 60: score += 3
    elif row['RiskScore'] > 30: score += 1
    if row['DebtToIncomeRatio'] > 0.4: score += 2
    elif row['DebtToIncomeRatio'] > 0.2: score += 1
    if row['PreviousLoanDefaults'] > 0: score += 3
    if score >= 7: return 'High Risk'
    elif score >= 3: return 'Medium Risk'
    else: return 'Low Risk'

df['RiskLevel_Final'] = df.apply(risk_level, axis=1)

def default_probability(row):
    prob = 0.1
    if row['CreditScore'] < 600: prob += 0.3
    elif row['CreditScore'] < 700: prob += 0.1
    if row['PreviousLoanDefaults'] > 0: prob += 0.35
    if row['DebtToIncomeRatio'] > 0.4: prob += 0.15
    if row['RiskScore'] > 60: prob += 0.1
    return min(prob, 1)

df['DefaultProbability'] = df.apply(default_probability, axis=1)

df['RiskLevel_Final'].value_counts()

## 6. 

In [ ]:
def financial_strength(row):
    score = 0
    if row['NetWorth'] > 0: score += 2
    if row['SavingsAccountBalance'] > 10000: score += 2
    if row['DebtToIncomeRatio'] < 0.3: score += 2
    if row['CreditScore'] > 700: score += 2
    if score >= 6: return 'Strong'
    elif score >= 3: return 'Moderate'
    else: return 'Weak'

df['FinancialStrength'] = df.apply(financial_strength, axis=1)
df['FinancialStrength'].value_counts()

## 7. 
Revenue = الفايدة الفعلية على القرض (TotalPayment - LoanAmount).
Expenses = افتراض 30% من الـ Revenue كتكلفة تشغيلية.

In [ ]:
df['TotalPayment'] = df['MonthlyLoanPayment'] * df['LoanDuration']
df['Revenue'] = df['TotalPayment'] - df['LoanAmount']
df['Expenses'] = df['Revenue'] * 0.30
df['Profit'] = df['Revenue'] - df['Expenses']

print("Financial Summary")
print("=" * 40)
print(f"Total Revenue:   ${df['Revenue'].sum():,.0f}")
print(f"Total Expenses:  ${df['Expenses'].sum():,.0f}")
print(f"Total Profit:    ${df['Profit'].sum():,.0f}")
print(f"Profit Margin:   {(df['Profit'].sum() / df['Revenue'].sum())*100:.1f}%")

## 8. 

In [ ]:
def income_level(income):
    if income < 40000: return 'Low Income'
    elif income < 80000: return 'Middle Income'
    else: return 'High Income'

df['IncomeLevel'] = df['AnnualIncome'].apply(income_level)

def customer_segment(row):
    age_tag = 'Young' if row['Age'] < 40 else 'Old'
    risk_tag = 'Risky' if row['RiskLevel_Final'] == 'High Risk' else 'Safe'
    return f"{age_tag} - {risk_tag}"

df['CustomerSegment'] = df.apply(customer_segment, axis=1)

print(df['IncomeLevel'].value_counts())
print()
print(df['CustomerSegment'].value_counts())

## 9.

In [ ]:
kpis = pd.DataFrame({
    "KPI": [
        "Total Revenue", "Total Profit", "Average Loan Amount",
        "Approval Rate (%)", "Default Rate (%)",
        "Average Risk Score", "Average Interest Rate"
    ],
    "Value": [
        df['Revenue'].sum(),
        df['Profit'].sum(),
        df['LoanAmount'].mean(),
        df['LoanApproved'].mean() * 100,
        df['PreviousLoanDefaults'].mean() * 100,
        df['RiskScore'].mean(),
        df['InterestRate'].mean()
    ]
})
kpis

## 10

In [ ]:
# DIM_Date
dim_date = df[['ApplicationDate','Year','Month','Quarter']].drop_duplicates().reset_index(drop=True)
dim_date.insert(0, 'DateID', range(1, len(dim_date) + 1))

# DIM_LoanPurpose
dim_purpose = df[['LoanPurpose']].drop_duplicates().reset_index(drop=True)
dim_purpose.insert(0, 'PurposeID', range(1, len(dim_purpose) + 1))

# DIM_CreditProfile
dim_credit = df[['CreditScore','CreditCategory','CreditCardUtilizationRate',
                  'NumberOfOpenCreditLines','NumberOfCreditInquiries',
                  'LengthOfCreditHistory','PaymentHistory',
                  'BankruptcyHistory','PreviousLoanDefaults']].drop_duplicates().reset_index(drop=True)
dim_credit.insert(0, 'CreditID', range(1, len(dim_credit) + 1))

# DIM_Risk
dim_risk = df[['RiskScore','RiskLevel_Final','DefaultProbability','FinancialStrength']].drop_duplicates().reset_index(drop=True)
dim_risk.insert(0, 'RiskID', range(1, len(dim_risk) + 1))

# DIM_Customer
dim_customer = df[['Age','AgeGroup','AnnualIncome','IncomeLevel','MonthlyIncome',
                    'EmploymentStatus','EducationLevel','MaritalStatus',
                    'NumberOfDependents','HomeOwnershipStatus','JobTenure',
                    'CustomerSegment']].drop_duplicates().reset_index(drop=True)
dim_customer.insert(0, 'CustomerID', range(1, len(dim_customer) + 1))

print(f"DIM_Date: {len(dim_date):,}")
print(f"DIM_LoanPurpose: {len(dim_purpose):,}")
print(f"DIM_CreditProfile: {len(dim_credit):,}")
print(f"DIM_Risk: {len(dim_risk):,}")
print(f"DIM_Customer: {len(dim_customer):,}")

## 11. 

In [ ]:
df = df.merge(dim_date[['DateID','ApplicationDate']], on='ApplicationDate', how='left')
df = df.merge(dim_purpose[['PurposeID','LoanPurpose']], on='LoanPurpose', how='left')

df = df.merge(
    dim_customer[['CustomerID','Age','AgeGroup','AnnualIncome','IncomeLevel','MonthlyIncome',
                  'EmploymentStatus','EducationLevel','MaritalStatus',
                  'NumberOfDependents','HomeOwnershipStatus','JobTenure','CustomerSegment']],
    on=['Age','AgeGroup','AnnualIncome','IncomeLevel','MonthlyIncome','EmploymentStatus',
        'EducationLevel','MaritalStatus','NumberOfDependents','HomeOwnershipStatus',
        'JobTenure','CustomerSegment'],
    how='left'
)

df = df.merge(
    dim_credit[['CreditID','CreditScore','CreditCategory','CreditCardUtilizationRate',
                'NumberOfOpenCreditLines','NumberOfCreditInquiries',
                'LengthOfCreditHistory','PaymentHistory','BankruptcyHistory','PreviousLoanDefaults']],
    on=['CreditScore','CreditCategory','CreditCardUtilizationRate','NumberOfOpenCreditLines',
        'NumberOfCreditInquiries','LengthOfCreditHistory','PaymentHistory',
        'BankruptcyHistory','PreviousLoanDefaults'],
    how='left'
)

df = df.merge(
    dim_risk[['RiskID','RiskScore','RiskLevel_Final','DefaultProbability','FinancialStrength']],
    on=['RiskScore','RiskLevel_Final','DefaultProbability','FinancialStrength'],
    how='left'
)

# Add LoanID + remove any duplicate matches from merges
df.insert(0, 'LoanID', range(1, len(df) + 1))
print("Duplicate LoanIDs before cleanup:", df['LoanID'].duplicated().sum())
df = df.drop_duplicates(subset='LoanID', keep='first').reset_index(drop=True)

print(f"Final df shape: {df.shape}")
df[['LoanID','DateID','PurposeID','CustomerID','CreditID','RiskID']].head()

## 12.

In [ ]:
fact_loans = df[[
    'LoanID', 'DateID', 'CustomerID', 'PurposeID', 'CreditID', 'RiskID',
    'LoanAmount', 'LoanDuration', 'InterestRate', 'MonthlyLoanPayment',
    'DebtToIncomeRatio', 'TotalDebtToIncomeRatio',
    'TotalAssets', 'TotalLiabilities', 'NetWorth',
    'SavingsAccountBalance', 'CheckingAccountBalance',
    'Revenue', 'Expenses', 'Profit', 'LoanApproved'
]]

print(f"Fact_Loans: {len(fact_loans):,} rows, {len(fact_loans.columns)} columns")
fact_loans.head()

## 13. 

In [ ]:
dim_date.to_csv("Dim_Date.csv", index=False)
dim_customer.to_csv("Dim_Customer.csv", index=False)
dim_purpose.to_csv("Dim_LoanPurpose.csv", index=False)
dim_credit.to_csv("Dim_CreditProfile.csv", index=False)
dim_risk.to_csv("Dim_Risk.csv", index=False)
fact_loans.to_csv("Fact_Loans.csv", index=False)

print("CSV files saved.")

## 14. 

In [ ]:
from sqlalchemy import create_engine

SERVER = "MOAZ\\SQLEXPRESS"
DATABASE = "LoanDW"

engine = create_engine(
    f"mssql+pyodbc://{SERVER}/{DATABASE}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

with engine.connect() as conn:
    print("Connected successfully!")

## 15. 

In [ ]:
tables = {
    'Dim_Date': dim_date,
    'Dim_Customer': dim_customer,
    'Dim_LoanPurpose': dim_purpose,
    'Dim_CreditProfile': dim_credit,
    'Dim_Risk': dim_risk,
    'Fact_Loans': fact_loans,
}

for table_name, df_table in tables.items():
    df_table.to_sql(
        table_name,
        engine,
        if_exists='replace',
        index=False,
        chunksize=1000
    )
    print(f"Loaded {table_name}: {len(df_table):,} rows")

print("\nAll tables loaded successfully!")

## 16. 

In [ ]:
for t in ['Dim_Date','Dim_Customer','Dim_LoanPurpose','Dim_CreditProfile','Dim_Risk','Fact_Loans']:
    cnt = pd.read_sql(f"SELECT COUNT(*) AS cnt FROM {t}", engine)['cnt'][0]
    print(f"{t}: {cnt:,} rows")